# Глава 7. Дискретное преобразование Фурье

### Пономарев Т. Е. 5130901/30201

## Упражнение 7.2
В этой главе я показал, как можно представить дискретное преобразование Фурье (DFT) и обратное преобразование Фурье (DFT)
в виде умножения матриц.  Эти операции занимают время, пропорциональное
$N^2$, где $N$ — длина векторного ряда.  Этого достаточно
для многих задач, но существует более быстрый
алгоритм — быстрое преобразование Фурье (FFT), время выполнения которого
пропорционально $N \log N$.

Ключом к FFT является лемма Даниэльсона-Ланцоса:

$DFT(y)[n] = DFT(e)[n] + \exp(-2 \pi i n / N) DFT(o)[n]$

Где $ DFT(y)[n]$ — $n$-й элемент DFT от $y$; $e$ — четные элементы $y$, а $o$ — нечетные элементы $y$.

Эта лемма подсказывает рекурсивный алгоритм для DFT:

1. Имея массив волн $y$, разделите его на четные элементы $e$ и нечетные элементы $o$.

2. Вычислить ДФТ $e$ и $o$ с помощью рекурсивных вызовов.

3. Вычислить $DFT(y)$ для каждого значения $n$ с использованием леммы Даниэльсона-Ланцоша.

В качестве базового случая этой рекурсии можно дождаться, пока длина
$y$ станет равна 1.  В этом случае $DFT(y) = y$.  Или, если длина $y$
достаточно мала, можно вычислить её ДФТ с помощью умножения матриц,
возможно, используя заранее вычисленную матрицу.

Подсказка: я предлагаю реализовать этот алгоритм постепенно, начав
с версии, которая не является строго рекурсивной.  На шаге 2 вместо
рекурсивного вызова используйте `dft` или `np.fft.fft`.  Сделайте так, чтобы шаг 3 работал,
и убедитесь, что результаты совпадают с результатами других
реализаций.  Затем добавьте базовый случай и убедитесь, что он работает. И, наконец, замените шаг 2 на рекурсивные вызовы.





In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from thinkdsp import PI2

ys = [-0.5, 0.1, 0.7, -0.1] # тестовый пример и вычисление его БПФ
hs = np.fft.fft(ys)
print(hs)

[ 0.2+0.j  -1.2-0.2j  0.2+0.j  -1.2+0.2j]


In [4]:
def dft(ys):       # реализация ДПФ из книги
    N = len(ys)
    ts = np.arange(N) / N
    freqs = np.arange(N)
    args = np.outer(ts, freqs)
    M = np.exp(1j * PI2 * args)
    amps = M.conj().transpose().dot(ys)
    return amps

hs2 = dft(ys)
np.sum(np.abs(hs - hs2)) # тот же результат (дельта маленькая)

np.float64(5.864775846765962e-16)

In [5]:
# БПФ версии, которая разделяет входной массив и использует np.fft.fft для вычисления БПФ для его половин
def fft_norec(ys):
    N = len(ys)
    He = np.fft.fft(ys[::2])
    Ho = np.fft.fft(ys[1::2])
    
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    return np.tile(He, 2) + W * np.tile(Ho, 2)

hs3 = fft_norec(ys)
np.sum(np.abs(hs - hs3)) # тот же результат (дельта маленькая)

np.float64(0.0)

In [6]:
# заменим np.fft.fft рекурсивными вызовами и добавим базовый случай
def fft(ys):
    N = len(ys)
    if N == 1:
        return ys
    
    He = fft(ys[::2])
    Ho = fft(ys[1::2])
    
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    return np.tile(He, 2) + W * np.tile(Ho, 2)

hs4 = fft(ys)
np.sum(np.abs(hs - hs4)) # тот же результат (дельта маленькая)

np.float64(1.6653345369377348e-16)

Эта реализация БПФ занимает время, пропорциональное $n \log n$.  Она также занимает объем памяти, пропорциональный $n \log n$, и тратит некоторое время на создание и копирование массивов